# Creating visualization dataframes 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from collections import Counter
from datetime import datetime


In [9]:
df = pd.read_csv('../../data/processed/aml_clean.csv')

In [ ]:
df['country_mismatch'] = df['s_country'] != df['r_country']

## Transactions dataframe

This is the baseline dataframe resulting from the cleaning process, with a few addition that we considered to be relevant for visualizations but that would result noisy when creating the features for our machine learning process.

In [10]:
df_trans = df

In [13]:
df_trans['date'] = pd.to_datetime(df_trans['date'])
df_trans['time'] = pd.to_datetime(df_trans['time'], format='%H:%M:%S').dt.time
df_trans['hour'] = pd.to_datetime(df_trans['time'], format='%H:%M:%S').dt.hour
df_trans['day_of_week'] = df_trans['date'].dt.day_name()

In [ ]:
# defining a function to fetch exchange rates and convert amounts to EUR

currency_map = {
    "US Dollar": "USD",
    "Bitcoin": "BTC",
    "Euro": "EUR",
    "Australian Dollar": "AUD",
    "Yuan": "CNY",
    "Rupee": "INR",
    "Mexican Peso": "MXN",
    "Yen": "JPY",
    "UK Pound": "GBP",
    "Ruble": "RUB",
    "Canadian Dollar": "CAD",
    "Swiss Franc": "CHF",
    "Brazil Real": "BRL",
    "Saudi Riyal": "SAR",
    "Shekel": "ILS"
}

def _fetch_rate(date_str: str, from_currency: str, to_currency: str = "EUR") -> float:
    """Fetch a single exchange rate from Frankfurter API for a given date."""

    from_currency = currency_map.get(from_currency, from_currency)

    if from_currency == "BTC":
        return 20222.71  # Fixed rate for Bitcoin to EUR as of september 2022
    
    if from_currency == to_currency:
        return 1.0

    url = f"https://api.frankfurter.app/{date_str}"
    response = requests.get(url, params={"from": from_currency, "to": to_currency})
    response.raise_for_status()
    return response.json()["rates"][to_currency]

def add_eur_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds two EUR-converted columns to the merged AML dataframe:
      - amount_received_eur
      - amount_paid_eur

    Fetches each unique (date, currency) pair ONCE, then applies
    via dict lookup — no per-row API calls.
    """

    # The date column is datetime after clean_transactions() — convert to string for the API
    date_strings = df["date"].dt.strftime("%Y-%m-%d")

    received_pairs = set(zip(date_strings, df["receiving_currency"]))
    paid_pairs     = set(zip(date_strings, df["payment_currency"]))
    all_pairs      = received_pairs | paid_pairs

    rate_cache: dict[tuple, float] = {}
    for date_str, currency in all_pairs:
        try:
            rate_cache[(date_str, currency)] = _fetch_rate(date_str, currency)
        except Exception:
            rate_cache[(date_str, currency)] = 1.0

    df["amount_received_eur"] = df.apply(
        lambda row: round(
            row["amount_received"] * rate_cache[(row["date"].strftime("%Y-%m-%d"), row["receiving_currency"])], 2
        ),
        axis=1,
    )
    df["amount_paid_eur"] = df.apply(
        lambda row: round(
            row["amount_paid"] * rate_cache[(row["date"].strftime("%Y-%m-%d"), row["payment_currency"])], 2
        ),
        axis=1,
    )

    return df

In [27]:
df_trans = add_eur_columns(df_trans)

## Accounts dataframe

This dataframe centers around the people's (account's) behavior. We decided to create it in the visualization step as we believed it could create interesting statistics and graphs to add to our dashboard.

In [29]:
df_accounts = df.groupby('s_entity_id').agg(
    entity_name=('s_entity_name', 'first'),
    entity_type=('s_entity_type', 'first'),
    bank=('s_bank_name', 'first'),
    country=('s_country', 'first'),
    n_transactions=('is_laundering', 'count'),
    n_laundering=('is_laundering', 'sum'),
    laundering_rate=('is_laundering', 'mean'),
    total_sent=('amount_paid', 'sum'),
    avg_sent=('amount_paid', 'mean'),
    n_unique_recipients=('r_entity_id', 'nunique'),
    n_countries_sent_to=('r_country', 'nunique'),
).reset_index()

print(df_accounts.shape)
print(df_accounts.head())

(160369, 12)
  s_entity_id              entity_name          entity_type  \
0   8000488C0          Partnership #37          Partnership   
1   800048940  Sole Proprietorship #33  Sole Proprietorship   
2   8000489A0        Partnership #2948          Partnership   
3   800048A00          Corporation #15          Corporation   
4   800048A60         Corporation #280          Corporation   

                       bank country  n_transactions  n_laundering  \
0     First Bank of Madison     USA             174             0   
1  National Bank of Laramie     USA             365             0   
2  National Bank of Laramie     USA             433             0   
3  National Bank of Laramie     USA             500             0   
4  National Bank of Laramie     USA             245             0   

   laundering_rate   total_sent       avg_sent  n_unique_recipients  \
0              0.0   1413666.64    8124.520920                   14   
1              0.0   7198610.48   19722.220493     

## Currency dataframe

This dataframe centers around the way money (currency) "behaves" trhoughout the laundering data. We decided to create it in the visualization step as we believed it could create interesting statistics and graphs to add to our dashboard.

In [28]:
# Money-led dataframe — one row per currency
df_money = df.groupby('payment_currency').agg(
    n_transactions=('amount_paid', 'count'),
    total_volume=('amount_paid', 'sum'),
    avg_transaction=('amount_paid', 'mean'),
    laundering_rate=('is_laundering', 'mean'),
    n_countries=('s_country', 'nunique'),
    n_cross_border=('country_mismatch', 'sum'),
).reset_index()

print(df_money.shape)
print(df_money.head())

(15, 7)
    payment_currency  n_transactions  total_volume  avg_transaction  \
0  Australian Dollar          136787  4.690341e+10     3.428938e+05   
1            Bitcoin          146066  3.039528e+06     2.080927e+01   
2        Brazil Real           70703  3.099436e+11     4.383740e+06   
3    Canadian Dollar          140042  8.095966e+10     5.781099e+05   
4               Euro         1168332  3.074193e+11     2.631266e+05   

   laundering_rate  n_countries  n_cross_border  
0         0.000928           24           53144  
1         0.000383           29           44630  
2         0.000806           21           26803  
3         0.000914           28           39346  
4         0.001174           35          896170  
